# Filterbank RFI cleaning

This notebook writes a new SIGPROC `.fil` file and never modifies the source.
Each complete 256-sample block is classified channel by channel with either the
legacy-compatible 1D CNN or the saved scikit-learn MLP. The final incomplete
block is copied unchanged.

`MODEL_KIND` and `REPLACEMENT` are included in the output filename. Before a
full run, configure the paths and inspect the preview. Writing is disabled
until `RUN_CLEANING = True`.

> [!warning]
> For noise replacement, define trusted RFI-free half-open intervals
> `(start_sample, stop_sample)` in `CLEAN_REFERENCE_RANGES`. One Gaussian
> distribution is estimated from these original samples before cleaning starts.


In [ ]:
def make_top3_features(segment_ct: np.ndarray) -> pd.DataFrame:
    values = np.asarray(segment_ct, dtype=np.float64)
    mean_o = values.mean(axis=1)
    std_o = values.std(axis=1)
    centered = values - mean_o[:, None]
    safe_std = np.where(std_o == 0, 1.0, std_o)
    skew_o = (centered ** 3).mean(axis=1) / safe_std ** 3
    return pd.DataFrame({"mean_o": mean_o, "std_o": std_o, "skew_o": skew_o})


def load_predictor():
    if MODEL_KIND == "cnn":
        if CNN_CHECKPOINT is None or not CNN_CHECKPOINT.is_file():
            raise FileNotFoundError("Set CNN_CHECKPOINT to an existing checkpoint.")
        device = torch.device(CNN_DEVICE)
        if device.type == "cuda" and not torch.cuda.is_available():
            raise RuntimeError("CNN_DEVICE is 'cuda', but CUDA is unavailable.")
        checkpoint = torch.load(CNN_CHECKPOINT, map_location=device)
        model = CNN1DRFI256Logits(dropout=0.5).to(device)
        model.load_state_dict(checkpoint.get("model_state_dict", checkpoint))
        model.eval()

        @torch.no_grad()
        def predict(segment_ct: np.ndarray):
            values = np.asarray(segment_ct, dtype=np.float32)
            maximum = values.max(axis=1, keepdims=True)
            maximum = np.where(maximum < 1e-8, 1.0, maximum)
            batch = torch.from_numpy(values / maximum).unsqueeze(1).to(device)
            probabilities = torch.sigmoid(model(batch)).cpu().numpy().astype(np.float32)
            return probabilities >= CNN_THRESHOLD, probabilities

        return predict, {
            "model_kind": "cnn", "checkpoint": str(CNN_CHECKPOINT),
            "threshold": float(CNN_THRESHOLD), "device": str(device),
            "normalization": "maximum per channel",
        }

    if MLP_BUNDLE is None or not MLP_BUNDLE.is_file():
        raise FileNotFoundError("Set MLP_BUNDLE to an existing MLP bundle.")
    bundle = joblib.load(MLP_BUNDLE)
    expected_features = ["mean_o", "std_o", "skew_o"]
    if list(bundle["feature_cols"]) != expected_features:
        raise ValueError(
            f"This cleaner expects {expected_features}; bundle has {bundle['feature_cols']}."
        )
    pipeline = bundle["pipeline"]
    threshold = float(bundle["threshold"])

    def predict(segment_ct: np.ndarray):
        probabilities = pipeline.predict_proba(make_top3_features(segment_ct))[:, 1]
        probabilities = probabilities.astype(np.float32)
        return probabilities >= threshold, probabilities

    return predict, {
        "model_kind": "mlp", "bundle": str(MLP_BUNDLE),
        "threshold": threshold, "feature_cols": expected_features,
    }


## Prepare the filterbank and the replacement distribution

`Your.get_data` supplies time-by-channel data. It is transposed only for model inference. For noise replacement, the stated clean regions are read from the original file and pooled into a single mean and standard deviation.

In [ ]:
def require_input_filterbank() -> None:
    if str(INPUT_FILTERBANK) == "/path/to/input.fil":
        raise ValueError("Set INPUT_FILTERBANK before running this cell.")
    if INPUT_FILTERBANK.suffix.lower() != ".fil" or not INPUT_FILTERBANK.is_file():
        raise FileNotFoundError(INPUT_FILTERBANK)


def validate_chunk(chunk: np.ndarray, n_channels: int) -> np.ndarray:
    chunk = np.asarray(chunk)
    if chunk.ndim != 2 or chunk.shape[1] != n_channels:
        raise ValueError(
            "Expected Your.get_data to return (n_samples, n_channels), "
            f"received {chunk.shape}."
        )
    return chunk


def estimate_reference_noise(reader: Your, n_channels: int) -> tuple[float, float]:
    if not CLEAN_REFERENCE_RANGES:
        raise ValueError("Noise replacement requires CLEAN_REFERENCE_RANGES.")
    total_count = total_sum = total_sum_sq = 0.0
    n_total = int(reader.your_header.nspectra)
    for start, stop in CLEAN_REFERENCE_RANGES:
        if not (0 <= start < stop <= n_total):
            raise ValueError(f"Invalid clean range {(start, stop)} for {n_total} samples.")
        for offset in range(start, stop, REFERENCE_READ_NSAMP):
            count = min(REFERENCE_READ_NSAMP, stop - offset)
            values = np.asarray(
                validate_chunk(reader.get_data(offset, count), n_channels), dtype=np.float64
            )
            total_count += values.size
            total_sum += values.sum()
            total_sum_sq += np.square(values).sum()
    mean = total_sum / total_count
    std = np.sqrt(max(total_sum_sq / total_count - mean ** 2, 0.0))
    return float(mean), max(float(std), 1e-6)


def cast_like_input(values: np.ndarray, dtype: np.dtype) -> np.ndarray:
    if np.issubdtype(dtype, np.integer):
        limits = np.iinfo(dtype)
        values = np.clip(np.rint(values), limits.min, limits.max)
    return values.astype(dtype, copy=False)


def replace_flagged(segment_ct, row_mask, rng, noise_mean, noise_std):
    cleaned = np.asarray(segment_ct).copy()
    if not row_mask.any():
        return cleaned
    if REPLACEMENT == "zero":
        cleaned[row_mask, :] = 0
        return cleaned
    noise = rng.normal(noise_mean, noise_std, size=(int(row_mask.sum()), cleaned.shape[1]))
    cleaned[row_mask, :] = cast_like_input(noise, cleaned.dtype)
    return cleaned


require_input_filterbank()
reader = Your(str(INPUT_FILTERBANK))
header = reader.your_header
if int(header.nifs) != 1:
    raise NotImplementedError("This notebook currently supports nifs = 1 only.")
if int(header.nbits) not in {8, 16, 32}:
    raise NotImplementedError("Supported input bit depths are 8, 16, and 32.")

n_channels = int(header.nchans)
n_total_samples = int(header.nspectra)
n_complete_segments, tail_nsamp = divmod(n_total_samples, SEGMENT_NSAMP)
probe_tc = validate_chunk(reader.get_data(0, min(SEGMENT_NSAMP, n_total_samples)), n_channels)
input_dtype = probe_tc.dtype
predict_segment, model_details = load_predictor()
probe_mask, probe_probabilities = predict_segment(probe_tc.T)
if probe_mask.shape != (n_channels,) or probe_probabilities.shape != (n_channels,):
    raise ValueError("The model must return one mask entry and probability per channel.")

noise_mean = noise_std = None
if REPLACEMENT == "noise":
    noise_mean, noise_std = estimate_reference_noise(reader, n_channels)

print(f"Samples: {n_total_samples}; channels: {n_channels}; dtype: {input_dtype}")
print(f"Complete blocks: {n_complete_segments}; unchanged tail: {tail_nsamp} samples")
print(f"Probe mask: {probe_mask.sum()} / {n_channels} channels")
if REPLACEMENT == "noise":
    print(f"Reference noise: mean={noise_mean:.6g}, std={noise_std:.6g}")


## Preview

Inspect one block before allowing the notebook to write a full output file. The middle panel shows the channels selected for replacement.

In [ ]:
if not 0 <= PREVIEW_SEGMENT_INDEX < n_complete_segments:
    raise ValueError(f"PREVIEW_SEGMENT_INDEX must be in [0, {n_complete_segments - 1}].")

preview_start = PREVIEW_SEGMENT_INDEX * SEGMENT_NSAMP
preview_tc = validate_chunk(reader.get_data(preview_start, SEGMENT_NSAMP), n_channels)
preview_ct = preview_tc.T
preview_mask, preview_probabilities = predict_segment(preview_ct)
preview_cleaned_ct = replace_flagged(
    preview_ct, preview_mask, np.random.default_rng(RANDOM_SEED), noise_mean, noise_std
)

figure, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
for axis, values, title in zip(
    axes,
    (preview_ct, np.repeat(preview_mask[:, None], SEGMENT_NSAMP, axis=1), preview_cleaned_ct),
    ("Original", "Selected channels", f"Cleaned ({REPLACEMENT})"),
):
    image = axis.imshow(values, aspect="auto", origin="upper", cmap="gray")
    axis.set(title=title, xlabel="Time sample")
axes[0].set_ylabel("Channel")
figure.colorbar(image, ax=axes, shrink=0.85)
figure.suptitle(
    f"Segment {PREVIEW_SEGMENT_INDEX}; {MODEL_KIND}; threshold={model_details['threshold']:.3f}"
)
figure.tight_layout()
plt.show()

print(f"Flagged channels: {preview_mask.sum()} / {n_channels}")


## Write the cleaned filterbank

The source header is copied to the new output. Existing outputs are refused, and the source file is never opened for writing. The summary, masks, and run manifest are written beside the cleaned filterbank.

In [ ]:
if not RUN_CLEANING:
    print("Full cleaning is disabled. Inspect the preview, then set RUN_CLEANING = True.")
else:
    if OUTPUT_FILTERBANK.resolve() == INPUT_FILTERBANK.resolve():
        raise ValueError("Output must not be the source filterbank.")
    outputs = (OUTPUT_FILTERBANK, SUMMARY_PATH, MASKS_PATH, MANIFEST_PATH)
    existing = [path for path in outputs if path.exists()]
    if existing:
        raise FileExistsError("Refusing to overwrite:\n" + "\n".join(map(str, existing)))

    OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
    output_header = SigprocFile(copy_hdr=SigprocFile(str(INPUT_FILTERBANK)))
    output_header.rawdatafile = str(OUTPUT_FILTERBANK)
    output_header.write_header(str(OUTPUT_FILTERBANK))

    rng = np.random.default_rng(RANDOM_SEED)
    masks = np.empty((n_complete_segments, n_channels), dtype=bool)
    summary_rows = []

    for segment_index in tqdm(range(n_complete_segments), desc="Cleaning"):
        start = segment_index * SEGMENT_NSAMP
        original_tc = validate_chunk(reader.get_data(start, SEGMENT_NSAMP), n_channels)
        original_ct = original_tc.T
        row_mask, probabilities = predict_segment(original_ct)
        row_mask = np.asarray(row_mask, dtype=bool)
        probabilities = np.asarray(probabilities, dtype=np.float32)
        cleaned_ct = replace_flagged(original_ct, row_mask, rng, noise_mean, noise_std)
        output_header.append_spectra(np.ascontiguousarray(cleaned_ct.T), str(OUTPUT_FILTERBANK))
        masks[segment_index] = row_mask
        summary_rows.append({
            "segment_index": segment_index, "start_sample": start,
            "n_masked": int(row_mask.sum()), "fraction_masked": float(row_mask.mean()),
            "probability_min": float(probabilities.min()),
            "probability_mean": float(probabilities.mean()),
            "probability_max": float(probabilities.max()),
        })

    if tail_nsamp:
        tail_tc = validate_chunk(
            reader.get_data(n_complete_segments * SEGMENT_NSAMP, tail_nsamp), n_channels
        )
        output_header.append_spectra(np.ascontiguousarray(tail_tc), str(OUTPUT_FILTERBANK))

    pd.DataFrame(summary_rows).to_csv(SUMMARY_PATH, index=False)
    np.savez_compressed(MASKS_PATH, masks=masks, segment_nsamp=SEGMENT_NSAMP)
    MANIFEST_PATH.write_text(json.dumps({
        "input_filterbank": str(INPUT_FILTERBANK),
        "output_filterbank": str(OUTPUT_FILTERBANK), "model": model_details,
        "replacement": REPLACEMENT, "clean_reference_ranges": CLEAN_REFERENCE_RANGES,
        "noise_mean": noise_mean, "noise_std": noise_std, "random_seed": RANDOM_SEED,
        "segment_nsamp": SEGMENT_NSAMP, "tail_nsamp_copied_unchanged": tail_nsamp,
    }, indent=2), encoding="utf-8")
    print(f"Created {OUTPUT_FILTERBANK}")
    print(f"Masked fraction: {masks.mean():.4%}")


## Validate the written output

Run this after cleaning. It reopens the output, checks defining header values and total sample count, and confirms that unmasked channels in the preview block are bitwise unchanged.

In [ ]:
if not OUTPUT_FILTERBANK.is_file():
    print("No output filterbank exists yet.")
else:
    output_reader = Your(str(OUTPUT_FILTERBANK))
    output_header = output_reader.your_header
    for field in ("nchans", "nbits", "nifs", "tsamp", "fch1", "foff", "tstart"):
        if getattr(header, field) != getattr(output_header, field):
            raise AssertionError(f"Header mismatch for {field}.")
    if int(output_header.nspectra) != n_total_samples:
        raise AssertionError("Output sample count does not match the source.")
    written_tc = validate_chunk(
        output_reader.get_data(preview_start, SEGMENT_NSAMP), n_channels
    )
    if not np.array_equal(written_tc[:, ~preview_mask], preview_tc[:, ~preview_mask]):
        raise AssertionError("At least one unmasked preview channel changed.")
    print("Validation passed: header, sample count, and unmasked preview channels agree.")
